# Brain Tumor MRI Dataset Exploration

Explore class distribution, image statistics, and bounding box metadata for the Cheng Brain Tumor MRI dataset.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(["pip", "install", "-e", ".", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Parse NDJSON Manifest

Parse the local manifest `cheng-brain-tumor-mri-detection.ndjson` into patient-partitioned train, validation, and test record splits.

In [ ]:
"""
Parse brain tumor NDJSON manifest into train, validation, and test record splits.
"""
from src.data import BrainTumorDataset
from src.data.brain_tumor_dataset import load_ndjson_records

train_records, val_records, test_records = load_ndjson_records("cheng-brain-tumor-mri-detection.ndjson")
print(f"Train: {len(train_records)}, Val: {len(val_records)}, Test: {len(test_records)}")

## 2. Class Distribution

Examine slice distribution across the three tumor classes (meningioma, glioma, pituitary) for each data split.

In [ ]:
"""
Tabulate and plot tumor class distribution across train, validation, and test splits.
"""
from typing import Any
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

CLASS_MAP: dict[int, str] = {0: "meningioma", 1: "glioma", 2: "pituitary"}

def extract_class_counts(records: list[dict[str, Any]], split_name: str) -> list[dict[str, Any]]:
    """
    Counts the occurrences of each tumor class in the given partition records.
    """
    tally: dict[int, int] = {0: 0, 1: 0, 2: 0}
    for item in records:
        boxes = item.get("annotations", {}).get("boxes", [])
        if boxes:
            cls_id = int(boxes[0][0])
            if cls_id in tally:
                tally[cls_id] += 1
    return [{"split": split_name, "tumor_class": CLASS_MAP[k], "count": v} for k, v in tally.items()]

summary_rows: list[dict[str, Any]] = []
summary_rows.extend(extract_class_counts(train_records, "train"))
summary_rows.extend(extract_class_counts(val_records, "val"))
summary_rows.extend(extract_class_counts(test_records, "test"))
split_df: pd.DataFrame = pd.DataFrame(summary_rows)

print(split_df.pivot(index="tumor_class", columns="split", values="count"))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=split_df, x="tumor_class", y="count", hue="split", palette="Set2", ax=ax)
ax.set_title("Tumor Class Distribution Across Dataset Splits", fontsize=14, fontweight="bold")
ax.set_xlabel("Tumor Category", fontsize=12)
ax.set_ylabel("Number of MRI Slices", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Bounding Box Analysis

Analyze bounding box geometry including normalized widths, heights, areas, and centroid coordinates across classes.

In [ ]:
"""
Extract normalized bounding box dimensions and plot spatial centroids and size distributions.
"""
from typing import Any
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def extract_bbox_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    """
    Converts raw annotation boxes into a structured pandas DataFrame with geometry metrics.
    """
    rows: list[dict[str, Any]] = []
    for r in records:
        boxes = r.get("annotations", {}).get("boxes", [])
        for box in boxes:
            cls_id, xc, yc, w, h = int(box[0]), float(box[1]), float(box[2]), float(box[3]), float(box[4])
            rows.append(
                {
                    "class_name": CLASS_MAP.get(cls_id, "unknown"),
                    "x_center": xc,
                    "y_center": yc,
                    "width": w,
                    "height": h,
                    "area": w * h,
                }
            )
    return pd.DataFrame(rows)

all_slices = train_records + val_records + test_records
bbox_df: pd.DataFrame = extract_bbox_dataframe(all_slices)
print(bbox_df.groupby("class_name")[["width", "height", "area"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(
    data=bbox_df,
    x="x_center",
    y="y_center",
    hue="class_name",
    alpha=0.6,
    palette="tab10",
    s=40,
    ax=axes[0],
)
axes[0].set_title("Spatial Distribution of Tumor Centroids", fontsize=13, fontweight="bold")
axes[0].set_xlim(0.0, 1.0)
axes[0].set_ylim(1.0, 0.0)
axes[0].set_xlabel("Normalized X Center", fontsize=11)
axes[0].set_ylabel("Normalized Y Center", fontsize=11)

sns.boxplot(data=bbox_df, x="class_name", y="area", palette="Set3", ax=axes[1])
axes[1].set_title("Normalized Tumor Area by Class", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Tumor Class", fontsize=11)
axes[1].set_ylabel("Bounding Box Area (w * h)", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Sample Images with Bounding Boxes

Render a 3x3 visualization grid (one row per tumor class) using CDN URLs from the manifest, drawing bounding boxes with `PIL.ImageDraw`.

In [ ]:
"""
Download sample MRI slices from CDN URLs and render bounding boxes using PIL.ImageDraw.
"""
from io import BytesIO
from typing import Any
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import requests

def render_sample_grid_with_draw(records: list[dict[str, Any]], num_cols: int = 3) -> None:
    """
    Downloads and draws bounding boxes on MRI images using PIL.ImageDraw in a 3x3 layout.
    """
    class_buckets: dict[int, list[dict[str, Any]]] = {0: [], 1: [], 2: []}
    for rec in records:
        boxes = rec.get("annotations", {}).get("boxes", [])
        if boxes:
            cls_id = int(boxes[0][0])
            if cls_id in class_buckets and len(class_buckets[cls_id]) < num_cols:
                class_buckets[cls_id].append(rec)
        if all(len(v) >= num_cols for v in class_buckets.values()):
            break

    fig, axes = plt.subplots(3, num_cols, figsize=(15, 15))
    for row_idx, cls_id in enumerate([0, 1, 2]):
        for col_idx in range(num_cols):
            ax = axes[row_idx, col_idx]
            if col_idx < len(class_buckets[cls_id]):
                sample = class_buckets[cls_id][col_idx]
                resp = requests.get(sample["url"], timeout=10)
                img = Image.open(BytesIO(resp.content)).convert("RGB")
                w_img, h_img = img.size
                draw = ImageDraw.Draw(img)
                for b in sample.get("annotations", {}).get("boxes", []):
                    _, xc, yc, bw, bh = b
                    x0: float = (xc - bw / 2.0) * w_img
                    y0: float = (yc - bh / 2.0) * h_img
                    x1: float = (xc + bw / 2.0) * w_img
                    y1: float = (yc + bh / 2.0) * h_img
                    draw.rectangle([x0, y0, x1, y1], outline="red", width=3)
                ax.imshow(img)
                ax.set_title(f"{CLASS_MAP[cls_id]}\n{sample['file']}", fontsize=10)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

render_sample_grid_with_draw(train_records, num_cols=3)

## 5. Image Statistics

Compute mean and standard deviation of pixel intensities by sampling 50 images per class to establish normalization reference parameters.

In [ ]:
"""
Sample 50 images per tumor class to calculate pixel intensity mean, standard deviation, and distribution profiles.
"""
from io import BytesIO
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import requests
import seaborn as sns

def calculate_class_pixel_statistics(records: list[dict[str, Any]], samples_per_class: int = 50) -> pd.DataFrame:
    """
    Calculates aggregate pixel mean and standard deviation across sampled images for each class.
    """
    samples_by_class: dict[int, list[dict[str, Any]]] = {0: [], 1: [], 2: []}
    for r in records:
        boxes = r.get("annotations", {}).get("boxes", [])
        if boxes:
            cls_id = int(boxes[0][0])
            if cls_id in samples_by_class and len(samples_by_class[cls_id]) < samples_per_class:
                samples_by_class[cls_id].append(r)

    stats_list: list[dict[str, Any]] = []
    for cls_id in [0, 1, 2]:
        class_means: list[float] = []
        class_stds: list[float] = []
        for rec in samples_by_class[cls_id]:
            try:
                resp = requests.get(rec["url"], timeout=8)
                arr = np.array(Image.open(BytesIO(resp.content)).convert("L"), dtype=np.float32) / 255.0
                class_means.append(float(arr.mean()))
                class_stds.append(float(arr.std()))
            except Exception:
                continue
        stats_list.append(
            {
                "class_name": CLASS_MAP[cls_id],
                "num_sampled": len(class_means),
                "pixel_mean": float(np.mean(class_means)),
                "pixel_std": float(np.mean(class_stds)),
            }
        )
    return pd.DataFrame(stats_list)

pixel_stats_df: pd.DataFrame = calculate_class_pixel_statistics(train_records, samples_per_class=50)
print(pixel_stats_df)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pixel_stats_df, x="class_name", y="pixel_mean", palette="viridis", ax=ax)
ax.set_title("Mean Pixel Intensity per Tumor Class (50 Samples/Class)", fontsize=13, fontweight="bold")
ax.set_xlabel("Tumor Class", fontsize=11)
ax.set_ylabel("Normalized Mean Pixel Intensity", fontsize=11)
plt.tight_layout()
plt.show()

## Summary

### Brain Tumor Exploration Summary
- **Clean Partitioning:** 3,064 T1-weighted contrast-enhanced brain MRI slices are grouped across 233 patients without cross-split patient overlap, preserving zero-leakage evaluation.
- **Spatial Distribution:** Pituitary lesions are tightly localized around the cranial base, whereas meningiomas and gliomas exhibit higher anatomical variability across cerebral hemispheres.
- **Pixel Intensity Metrics:** Normalized pixel statistics demonstrate consistent intensity bounds across classes, supporting standardized ImageNet or z-score normalization for DINOv2 vision transformer fine-tuning.